Defining the domain

In [ ]:
 #Defining $\sigma^* $
import numpy as np
import matplotlib.pyplot as plt
import torch

# grid resolution
N = 100

x = np.linspace(-0.5, 0.5, N)
y = np.linspace(-0.5, 0.5, N)
X, Y = np.meshgrid(x, y)

#Defining the domain and bondary


# domain limits
xmin, xmax = -0.5, 0.5
ymin, ymax = -0.5, 0.5

# grid spacing
hx = (xmax - xmin) / (N - 1)
hy = (ymax - ymin) / (N - 1)

# 1D grids
x = np.linspace(xmin, xmax, N)
y = np.linspace(ymin, ymax, N)

# 2D mesh
X, Y = np.meshgrid(x, y)

Defining boundary

In [ ]:
# Collect boundary grid points (ordered clockwise, no duplicate corners)
# Domain: square [-0.5, 0.5]^2 with grid spacing hx, hy from (x, y)

boundary_x = []
boundary_y = []

# 1. Bottom edge: left → right (y = ymin)
boundary_x.extend(x)
boundary_y.extend([ymin] * N)

# 2. Right edge: bottom → top (x = xmax), exclude corners already in bottom/top
boundary_x.extend([xmax] * (N - 2))
boundary_y.extend(y[1:-1])

# 3. Top edge: right → left (y = ymax)
boundary_x.extend(x[::-1])
boundary_y.extend([ymax] * N)

# 4. Left edge: top → bottom (x = xmin), exclude corners
boundary_x.extend([xmin] * (N - 2))
boundary_y.extend(y[-2:0:-1])

boundary_x = np.array(boundary_x)
boundary_y = np.array(boundary_y)

m = len(boundary_x)  # total boundary grid points
print("Total boundary points 4N-4:", m)

Defining L-shape sigma

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 100

x = np.linspace(-0.5, 0.5, N)
y = np.linspace(-0.5, 0.5, N)
X, Y = np.meshgrid(x, y)

# σ* = 100 on L-shape, 0 elsewhere
def in_L(x, y):
    in_h = (-0.25 <= x) & (x <= 0.25) & (0 <= y) & (y <= 0.25)
    in_v = (-0.25 <= x) & (x <= 0) & (-0.5 <= y) & (y <= 0.25)
    return in_h | in_v

Sigma_star = np.where(in_L(X, Y), 100.0, 1.0)

plt.figure()
plt.imshow(Sigma_star, extent=[xmin, xmax, ymin, ymax], origin='lower')
plt.colorbar(label=r"$\sigma^*$")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"$\sigma^*$: 100 on L-shape,1 elsewhere")
plt.gca().set_aspect("equal")
plt.show()

Uniform placement of source and censors

In [ ]:
# Every boundary grid point is either a source or a receiver; uniform and alternating (source → receiver → source → ...)
# m = 4*N - 4 total boundary points

m = len(boundary_x)  # 4*N - 4

n1_idx = np.arange(0, m, 2)   # sources at even boundary indices
n2_idx = np.arange(1, m, 2)   # receivers at odd boundary indices

n1 = len(n1_idx)
n2 = len(n2_idx)
n1_x = boundary_x[n1_idx]
n1_y = boundary_y[n1_idx]
n2_x = boundary_x[n2_idx]
n2_y = boundary_y[n2_idx]

assert n1 + n2 == m
assert len(np.intersect1d(n1_idx, n2_idx)) == 0

print("Number of sources (n_1):", n1)
print("Number of receivers (n_2):", n2)

Building F and G

In [ ]:
# F: n_1 × m — row i is delta at source i (1 at boundary index n1_idx[i])
F = np.zeros((n1, m))
for i in range(n1):
    F[i, n1_idx[i]] = 1.0

# G: n_2 × m — row j is delta at receiver j (1 at boundary index n2_idx[j])
G = np.zeros((n2, m))
for j in range(n2):
    G[j, n2_idx[j]] = 1.0

print("F shape:", F.shape, "  G shape:", G.shape)

## Perturbation: four separated squares (same size as original)

Original single square: $[-0.2,0]\times[-0.2,0]$, amplitude $10^{-3}$, side length $0.2$.

Four copies of that square, placed separately near the corners of $[-0.5,0.5]^2$.


In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1) Four-square delta_sigma (each 0.2 x 0.2, amplitude 1e-3)
# ------------------------------------------------------------
xmin, xmax, ymin, ymax = -0.5, 0.5, -0.5, 0.5
h = (xmax - xmin) / (N - 1)

x = np.linspace(xmin, xmax, N)
y = np.linspace(ymin, ymax, N)
X, Y = np.meshgrid(x, y)

delta_sigma_const = 1e-3
side = 0.2  # same side length as the original single square

# Four separated 0.2x0.2 squares (same size as original), one per corner region
square_side = 0.2
squares = [
    (-0.45, -0.25, -0.45, -0.25),  # bottom-left
    ( 0.25,  0.45, -0.45, -0.25),  # bottom-right
    (-0.45, -0.25,  0.25,  0.45),  # top-left
    ( 0.25,  0.45,  0.25,  0.45),  # top-right
]

support = np.zeros_like(X, dtype=bool)
for x0, x1, y0, y1 in squares:
    support |= (X >= x0) & (X <= x1) & (Y >= y0) & (Y <= y1)

delta_sigma = np.where(support, delta_sigma_const, 0.0).astype(np.float64)
Sigma = Sigma_star + delta_sigma

print("number of perturbed pixels:", int(support.sum()))
print("fraction support:", float(np.mean(support)))
print("delta_sigma min/max:", delta_sigma.min(), delta_sigma.max())
print("Sigma min/max:", Sigma.min(), Sigma.max())

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(delta_sigma, extent=[xmin, xmax, ymin, ymax], origin="lower", cmap="viridis")
ax.set_title(r"$\delta\sigma$: four separated $0.2\times0.2$ squares")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
plt.colorbar(im, ax=ax)
plt.show()

# ------------------------------------------------------------
# 2) FEM utilities (same grid / boundary order as Full Matrix notebook)
# ------------------------------------------------------------
nodes = np.array([(x[i], y[j]) for j in range(N) for i in range(N)])
n_nodes = N * N

def idx(i, j):
    return j * N + i

triangles = []
for j in range(N - 1):
    for i in range(N - 1):
        n1_, n2_ = idx(i, j), idx(i + 1, j)
        n3_, n4_ = idx(i, j + 1), idx(i + 1, j + 1)
        triangles.append([n1_, n2_, n4_])
        triangles.append([n1_, n4_, n3_])

boundary_ij = (
    [(i, 0) for i in range(N)]
    + [(N - 1, j) for j in range(1, N - 1)]
    + [(i, N - 1) for i in range(N - 1, -1, -1)]
    + [(0, j) for j in range(N - 2, 0, -1)]
)
boundary_dofs = np.array([idx(i, j) for (i, j) in boundary_ij], dtype=np.int64)

m_bd = len(boundary_x)
assert len(boundary_dofs) == m_bd
assert F.shape[1] == m_bd and G.shape[1] == m_bd

is_boundary = np.zeros(n_nodes, dtype=bool)
is_boundary[boundary_dofs] = True
interior = np.where(~is_boundary)[0]


def assemble_K_from_sigma(sigma_grid):
    K = sp.lil_matrix((n_nodes, n_nodes))
    for tri in triangles:
        pts = nodes[tri]
        x0, x1, x2 = pts
        B = np.array([
            [x1[0] - x0[0], x2[0] - x0[0]],
            [x1[1] - x0[1], x2[1] - x0[1]],
        ])
        area = abs(np.linalg.det(B)) / 2.0
        C = np.linalg.inv(B).T
        grads = np.array([[-1, -1], [1, 0], [0, 1]]) @ C

        center = pts.mean(axis=0)
        ic = int(round((center[0] - xmin) / h))
        jc = int(round((center[1] - ymin) / h))
        ic = np.clip(ic, 0, N - 1)
        jc = np.clip(jc, 0, N - 1)
        sigma_val = sigma_grid[jc, ic]

        for a in range(3):
            for b_ in range(3):
                K[tri[a], tri[b_]] += sigma_val * area * np.dot(grads[a], grads[b_])
    return K.tocsr()


def solve_all_u_for_sigma(sigma_grid, Fmat):
    K = assemble_K_from_sigma(sigma_grid)
    Aii = K[interior][:, interior]
    Aib = K[interior][:, boundary_dofs]
    solve_II = spla.splu(Aii.tocsc()).solve

    def solve_one_bc(bc_vals):
        bc_vals = np.asarray(bc_vals, dtype=np.float64).flatten()
        rhs = -Aib @ bc_vals
        uI = solve_II(rhs)
        u = np.zeros(n_nodes, dtype=np.float64)
        u[interior] = uI
        u[boundary_dofs] = bc_vals
        return u.reshape(N, N)

    return [solve_one_bc(Fmat[i, :]) for i in range(Fmat.shape[0])]


# ------------------------------------------------------------
# 3) Solve u* and u with the same F
# ------------------------------------------------------------
u_star_i_list = solve_all_u_for_sigma(Sigma_star, F)
u_i_list = solve_all_u_for_sigma(Sigma, F)

print("Solved u* count:", len(u_star_i_list), "  u count:", len(u_i_list))


# ------------------------------------------------------------
# 4) Outward normal derivative on boundary
# ------------------------------------------------------------
def boundary_dn_u(u):
    q = []
    j = 0
    for i in range(N):
        q.append(-(u[j + 1, i] - u[j, i]) / h)
    i = N - 1
    for j in range(1, N - 1):
        q.append((u[j, i] - u[j, i - 1]) / h)
    j = N - 1
    for i in range(N - 1, -1, -1):
        q.append((u[j, i] - u[j - 1, i]) / h)
    i = 0
    for j in range(N - 2, 0, -1):
        q.append(-(u[j, i + 1] - u[j, i]) / h)
    return np.asarray(q, dtype=np.float64)


dn_u_all = np.array([boundary_dn_u(u) for u in u_i_list])
dn_u_star_all = np.array([boundary_dn_u(u) for u in u_star_i_list])

print("dn_u_all shape:", dn_u_all.shape)
print("dn_u_star_all shape:", dn_u_star_all.shape)


## Assemble \(b\) (same formula as Full Matrix notebook, cell 21)

In [ ]:
import numpy as np


def sigma_on_boundary(sigma_grid):
    vals = []
    j = 0
    for i in range(N):
        vals.append(sigma_grid[j, i])
    i = N - 1
    for j in range(1, N - 1):
        vals.append(sigma_grid[j, i])
    j = N - 1
    for i in range(N - 1, -1, -1):
        vals.append(sigma_grid[j, i])
    i = 0
    for j in range(N - 2, 0, -1):
        vals.append(sigma_grid[j, i])
    return np.asarray(vals, dtype=np.float64)


sigma_bd = sigma_on_boundary(Sigma)
sigma_star_bd = sigma_on_boundary(Sigma_star)

sigma_recv = sigma_bd[n2_idx]
sigma_star_recv = sigma_star_bd[n2_idx]

LHS_mat = dn_u_all[:, n2_idx] * sigma_recv[None, :] \
        - dn_u_star_all[:, n2_idx] * sigma_star_recv[None, :]

b = LHS_mat.ravel()

print("LHS_mat shape:", LHS_mat.shape)
print("b shape:", b.shape)
print("||b||:", np.linalg.norm(b))
